In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import zipfile

print("=== GIAI ĐOẠN 1: NẠP MA TRẬN XÁC SUẤT CỦA 3 MODELS ===")
STANCE2ID = {"Against": 0, "Favor": 1, "None": 2}
ID2LABEL = {0: "Against", 1: "Favor", 2: "None"}

train_df = pd.read_csv("../data/train.csv", keep_default_na=False)
true_train_labels = train_df["stance"].str.strip().map(STANCE2ID).fillna(2).astype(int).values

dev_df = pd.read_csv("../data/dev.csv", keep_default_na=False)
true_dev_labels = dev_df["stance"].str.strip().map(STANCE2ID).fillna(2).astype(int).values

# Nạp OOF Probs
marbert_oof = np.load("../model/marbert_oof_probs.npy")
arabert_oof = np.load("../model/arabert_oof_probs.npy")

# Nạp Test Probs
marbert_test = np.load("../model/marbert_test_probs.npy")
arabert_test = np.load("../model/arabert_test_probs.npy")

print("\n=== GIAI ĐOẠN 2: TRỘN XÁC SUẤT BẰNG WEIGHTED SOFT VOTING ===")
W_MAR, W_ARA = 0.6, 0.4

ensemble_oof = (W_MAR * marbert_oof) + (W_ARA * arabert_oof)
ensemble_test = (W_MAR * marbert_test) + (W_ARA * arabert_test)

print("\n=== GIAI ĐOẠN 3: DÒ NGƯỠNG TARGET-WISE TRÊN MẢNG ENSEMBLE TỔNG ===")
def optimize_target_margin(probs, labels):
    best_score = 0
    best_th = {"Against": 0.33, "Favor": 0.33}
    
    # Quét lưới ngưỡng siêu mịn
    for th_against in np.arange(0.1, 0.7, 0.01):
        for th_favor in np.arange(0.1, 0.6, 0.01):
            preds = []
            for p in probs:
                margin = p - np.array([th_against, th_favor, 0.0])
                preds.append(2 if margin.max() < 0 else int(np.argmax(margin)))
            
            f_ag = f1_score(labels, preds, labels=[0], average="macro")
            f_fav = f1_score(labels, preds, labels=[1], average="macro")
            score = (f_fav + f_ag) / 2.0
            
            if score > best_score:
                best_score = score
                best_th = {"Against": th_against, "Favor": th_favor}
    return best_th, best_score

target_thresholds = {}
oof_preds = np.zeros(len(train_df))

for t in train_df['target'].unique():
    idx = train_df['target'] == t
    th, score = optimize_target_margin(ensemble_oof[idx], true_train_labels[idx])
    target_thresholds[t] = th
    print(f"🎯 Target: {t: <25} | OOF Favg2: {score:.4f} | Ngưỡng: {th}")
    
    for i in np.where(idx)[0]:
        margin = ensemble_oof[i] - np.array([th['Against'], th['Favor'], 0.0])
        oof_preds[i] = 2 if margin.max() < 0 else int(np.argmax(margin))

global_oof_favg2 = (f1_score(true_train_labels, oof_preds, labels=[0], average="macro") + 
                    f1_score(true_train_labels, oof_preds, labels=[1], average="macro")) / 2.0
print(f"\n🔥 TỔNG HỢP OOF FAVG2 CỦA MEGA-ENSEMBLE: {global_oof_favg2:.4f}")

print("\n=== GIAI ĐOẠN 4: INFERENCE LÊN DEV VÀ ĐÓNG GÓI ===")
final_preds_tuned = []
for i, p in enumerate(ensemble_test):
    t = dev_df['target'].iloc[i]
    th = target_thresholds.get(t, {"Against": 0.33, "Favor": 0.33})
    margin = p - np.array([th['Against'], th['Favor'], 0.0])
    final_preds_tuned.append(2 if margin.max() < 0 else int(np.argmax(margin)))

dev_f_ag = f1_score(true_dev_labels, final_preds_tuned, labels=[0], average="macro")
dev_f_fav = f1_score(true_dev_labels, final_preds_tuned, labels=[1], average="macro")
print(f"📊 ĐIỂM MEGA-ENSEMBLE CHỐT (LEADERBOARD ESTIMATE): {(dev_f_ag + dev_f_fav) / 2.0:.4f}")

# Xuất file nộp bài định dạng chuẩn
predicted_labels = [ID2LABEL[pred] for pred in final_preds_tuned]
with open("submission_mega_ensemble.txt", "w", encoding="utf-8") as f:
    for label in predicted_labels:
        f.write(label + "\n")

with zipfile.ZipFile("submission_mega_ensemble.zip", "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write("submission_mega_ensemble.txt")

print("\n✅ Đã xuất file: 'submission_mega_ensemble.zip'.")

=== GIAI ĐOẠN 1: NẠP MA TRẬN XÁC SUẤT CỦA 3 MODELS ===

=== GIAI ĐOẠN 2: TRỘN XÁC SUẤT BẰNG WEIGHTED SOFT VOTING ===

=== GIAI ĐOẠN 3: DÒ NGƯỠNG TARGET-WISE TRÊN MẢNG ENSEMBLE TỔNG ===
🎯 Target: Women empowerment         | OOF Favg2: 0.8579 | Ngưỡng: {'Against': np.float64(0.2799999999999999), 'Favor': np.float64(0.13999999999999999)}
🎯 Target: Covid Vaccine             | OOF Favg2: 0.8153 | Ngưỡng: {'Against': np.float64(0.21999999999999995), 'Favor': np.float64(0.12)}
🎯 Target: Digital Transformation    | OOF Favg2: 0.7234 | Ngưỡng: {'Against': np.float64(0.1), 'Favor': np.float64(0.2799999999999999)}

🔥 TỔNG HỢP OOF FAVG2 CỦA MEGA-ENSEMBLE: 0.8298

=== GIAI ĐOẠN 4: INFERENCE LÊN DEV VÀ ĐÓNG GÓI ===
📊 ĐIỂM MEGA-ENSEMBLE CHỐT (LEADERBOARD ESTIMATE): 0.8366

✅ Đã xuất file: 'submission_mega_ensemble.zip'.
